# 3 - Metadaten abholen

### Metadaten aus Alma (SRU, marcxml)

Mit der MMS ID werden die MARC-Daten via SRU aus Alma extrahiert und abgespeichert unter metadata/{signature}/signature.xml. Für gocfl create müssen die Metadaten pro Objekt in einem eigenen Ordner liegen.

### METS-Daten

In den ZIP-Kapseln ist jeweils eine METS-Datei vorhanden aus dem Original-Digitalisierungsprojekt.

### Weitere Metadaten

z.B. IIIF (e-manuscripta), Datacite (LARA), Aalto, etc. 

In [2]:
import requests
import json
from datetime import datetime
from pathlib import Path
import config

# which metadata is available:
marc = config.marcxml
marc_url = config.marcxml_baseurl

iiif = config.iiif
iiif_url = config.iiif_baseurl

datacite = config.datacite
datacite_url = config.datacite_baseurl
datacite_set = config.datacite_set

# general config:

urn = config.ingest_workflow
collection = config.collection_id
md_path = f'{collection}/{config.metadata_path}'
org_id = config.organisation_id

input_file = f"{collection}/{config.files_path}/{collection}_complete_set.json"

with open(input_file) as data_file:    
    data = json.load(data_file)
    for value in data:
        
        foldername = value["signature"][(len(org_id)+1):]
        identifiers = {}
        for item in value["identifiers"]:
            # split identifiers in dict
            [key, value] = item.split(':',1)
            identifiers[key] = value
        #print(identifiers)
        
        # create new directory for each signature (ignore, if it already exists)
        Path(f'{md_path}/{foldername}').mkdir(parents=True, exist_ok=True)
        
        # check for metadata settings
        
        # marcxml data:
        
        if marc == 'true':
            
            # get mms_id
            mmsid = identifiers['mmsid']
            
            # get SRU response
            query = marc_url+mmsid
            response = requests.get(query)
            if response.status_code != 200:
                raise Exception(f"SRU request failed with status code {response.status_code}")

            # Save the response content as xml to a new directory
            marcxmlfile = f"{md_path}/{foldername}/{mmsid}.xml"

            with open(marcxmlfile, 'wb') as file:
                file.write(response.content)
                print(f"\nRecord with ID {mmsid} saved as {marcxmlfile}\n---")
            
        
        # IIIF metadata
        
        if iiif == 'true':
            
            # get external id
            iiif_id = identifiers[urn]
            
            # get query for IIIF manifest
            query = iiif_url+iiif_id+'/manifest'

            response = requests.get(query)
            if response.status_code != 200:
                raise Exception(f"IIIF request failed with status code {response.status_code}")

            # Save the response content as xml to a new directory
            iiiffile = f"{md_path}/{foldername}/{iiif_id}-manifest.json"

            with open(iiiffile, 'wb') as file:
                file.write(response.content)
                print(f"\nRecord with ID {iiif_id} saved as {iiiffile}\n---")
            

        
print("Finished at ",datetime.today().strftime('%Y-%m-%d %H:%M:%S'))


Record with ID 9914289852805505 saved as zhb_e-manuscripta/metadata/10_7891_e-manuscripta-108732/9914289852805505.xml
---

Record with ID 3120805 saved as zhb_e-manuscripta/metadata/10_7891_e-manuscripta-108732/3120805-manifest.json
---

Record with ID 9914249335105505 saved as zhb_e-manuscripta/metadata/10_7891_e-manuscripta-24104/9914249335105505.xml
---

Record with ID 1060335 saved as zhb_e-manuscripta/metadata/10_7891_e-manuscripta-24104/1060335-manifest.json
---

Record with ID 9914249334005505 saved as zhb_e-manuscripta/metadata/10_7891_e-manuscripta-24479/9914249334005505.xml
---

Record with ID 1065627 saved as zhb_e-manuscripta/metadata/10_7891_e-manuscripta-24479/1065627-manifest.json
---

Record with ID 9914249332505505 saved as zhb_e-manuscripta/metadata/10_7891_e-manuscripta-24545/9914249332505505.xml
---

Record with ID 1070517 saved as zhb_e-manuscripta/metadata/10_7891_e-manuscripta-24545/1070517-manifest.json
---

Record with ID 9914249332405505 saved as zhb_e-manusc